# import and define

In [37]:
#NOTE load model code 
"""

I think i just realized that models loaded on M1 cause there to be an issue saving it on mrx link


"""

import os
import torch
import platform
from torchvision import models
from torchsummary import summary

def get_device(skip_apple_silicon=False) -> (str, str):
    """
    Determines the device to use based on availability.
    Returns:
        device (str): The torch device string.
        device_name (str): A human-friendly name for the device.
    """
    if torch.cuda.is_available():
        device = "cuda"
        device_name = f"CUDA GPU ({torch.cuda.get_device_name(0)})"
    elif platform.processor() == 'arm' and platform.system() == 'Darwin' and not skip_apple_silicon:
        device = "mps"  # For Apple Silicon (using MPS backend)
        device_name = "Apple Silicon (MPS)"
    else:
        device = "cpu"
        device_name = "CPU"
    return device, device_name

def load_or_save_resnet18(device: str, model_dir: str = "model", filename: str = "resnet18.pth") -> torch.nn.Module:
    """
    Loads the pretrained ResNet-18 model from disk if it exists.
    Otherwise, downloads, saves, and returns the model.
    """
    os.makedirs(model_dir, exist_ok=True)
    model_path = os.path.join(model_dir, filename)

    if os.path.exists(model_path):
        print("Loading ResNet-18 model from disk...")
        resnet18 = models.resnet18(pretrained=False)  # Instantiate without pretrained weights
        resnet18.load_state_dict(torch.load(model_path, map_location=device))
    else:
        print("Downloading pretrained ResNet-18 model...")
        resnet18 = models.resnet18(pretrained=True)
        torch.save(resnet18.state_dict(), model_path)
    
    return resnet18.to(device)



import torch

def list_layers(model: torch.nn.Module) -> None:
    """
    Lists the names of all modules in the model.
    """
    print("Available layers:")
    for name, module in model.named_modules():
        # Avoid printing the top-level model twice.
        if name:
            print(name)

def get_activation(model: torch.nn.Module, layer_name: str, input_tensor: torch.Tensor) -> torch.Tensor:
    """
    Extracts activations from a specified layer in the model using a forward hook.

    Args:
        model: The neural network (e.g., ResNet-18).
        layer_name: The name of the layer to hook (e.g., 'layer4').
        input_tensor: A batch of input images, shape (batch_size, 3, 224, 224).

    Returns:
        The activation tensor from the specified layer.
    """
    activations = {}
    
    def hook_fn(module, input, output):
        activations[layer_name] = output.detach()
    
    # Get the target module; if not found, raise an error.
    try:
        target_module = dict(model.named_modules())[layer_name]
    except KeyError:
        raise ValueError(f"Layer '{layer_name}' not found. Use list_layers(model) to see available layers.")
    
    # Register the hook, run a forward pass, and remove the hook.
    hook_handle = target_module.register_forward_hook(hook_fn)
    _ = model(input_tensor)
    hook_handle.remove()
    
    return activations[layer_name]


    
def compute_activation_similarity(activations: torch.Tensor) -> torch.Tensor:
    """
    Computes a similarity (correlation) matrix for a batch of activations.
    
    Args:
        activations: Tensor of shape [batch_size, channels, height, width]
    
    Returns:
        similarity_matrix: Tensor of shape [batch_size, batch_size]
    """
    batch_size = activations.size(0)
    # Flatten each activation tensor into a vector
    flat_activations = activations.view(batch_size, -1)  # shape: [batch_size, channels*height*width]
    # Compute the similarity matrix as a dot product between flattened vectors
    similarity_matrix = torch.matmul(flat_activations, flat_activations.t())
    return similarity_matrix






# load model print layers

In [ ]:
# NOTE load the model 

# Set device info
device, device_name = get_device(skip_apple_silicon=True)
print(f"Using device: {device_name}")

# Load the ResNet-18 model (download if not already saved)
resnet18 = load_or_save_resnet18(device)


# List available layers to help select the desired one:
list_layers(resnet18)




# get sample data

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torchvision.transforms as transforms
# import torchvision.datasets as datasets
# from torch.utils.data import DataLoader

# data_dir = "./data"
# # Check if CIFAR-10 dataset already exists
# cifar_folder = os.path.join(data_dir, "cifar-10-batches-py")
# download = not os.path.exists(cifar_folder)

# # Define transformations
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),  # ResNet18 expects 224x224 images
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], 
#                          std=[0.229, 0.224, 0.225]),  # Standard normalization
# ])

# # Load dataset and extract a sample batch
# dataset = datasets.CIFAR10(root=data_dir, train=False, download=download, transform=transform)
# dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# # Get a batch of real images & labels
# sample_input, sample_labels = next(iter(dataloader))
# sample_input = sample_input.to(device)  # Move to the correct device
# sample_labels = sample_labels.to(device)  # Move to device

# print("Sample Input Shape:", sample_input.shape)
# print("Sample Labels Shape:", sample_labels.shape)



import os
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

# Set device (ensure this is defined in your environment)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = "./data"
cifar_folder = os.path.join(data_dir, "cifar-10-batches-py")
download = not os.path.exists(cifar_folder)

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet18 expects 224x224 images
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]),  # Standard normalization
])

# Load dataset without shuffling to maintain order
dataset = datasets.CIFAR10(root=data_dir, train=False, download=download, transform=transform)
classes = dataset.classes  # List of class names in order

# Choose first 4 classes to get a total of 8 samples (2 per class)
desired_classes = classes[:4]  
samples_per_class = {cls: [] for cls in desired_classes}

# Iterate through dataset to collect 2 samples per desired class (non-random)
for img, label in dataset:
    class_name = classes[label]
    if class_name in desired_classes and len(samples_per_class[class_name]) < 2:
        samples_per_class[class_name].append((img, label))
    # Stop when 2 samples for each desired class are collected
    if all(len(samples) == 2 for samples in samples_per_class.values()):
        break

# Combine the samples in order (2 samples per class in the order of desired_classes)
ordered_samples = []
ordered_labels = []
ordered_class_names = []
for cls in desired_classes:
    for img, label in samples_per_class[cls]:
        ordered_samples.append(img)
        ordered_labels.append(label)
        ordered_class_names.append(cls)

# Create sample_input and sample_labels exactly as specified and load them to device
sample_input = torch.stack(ordered_samples).to(device)
sample_labels = torch.tensor(ordered_labels).to(device)

# Display the images with their class names
plt.figure(figsize=(10, 5))
for i, (img, cls_name) in enumerate(zip(ordered_samples, ordered_class_names)):
    # Convert tensor (C x H x W) to numpy array (H x W x C)
    npimg = img.cpu().permute(1, 2, 0).numpy()
    # Inverse normalization for visualization
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    npimg = std * npimg + mean
    npimg = np.clip(npimg, 0, 1)
    plt.subplot(2, 4, i+1)
    plt.imshow(npimg)
    plt.title(str(i)+' '+cls_name)
    plt.axis('off')
plt.tight_layout()
plt.show()




# activations

In [ ]:
# Sample input batch
# sample_input = torch.randn(8, 3, 224, 224).to(device)  # batch of 8 images

# Extract activations from a chosen layer (e.g., 'layer4')
layer4_activations = get_activation(resnet18, 'layer4', sample_input)
print("layer4 activations shape:", layer4_activations.shape)


# Compute and display the activation similarity matrix
activation_similarity = compute_activation_similarity(layer4_activations)
print("Activation similarity matrix shape:", activation_similarity.shape)




# V1 ENTK

In [ ]:




import torch
import torch.nn as nn

def compute_empirical_ntk(
    model: torch.nn.Module,
    inputs: torch.Tensor,
    labels: torch.Tensor,
    criterion: torch.nn.modules.loss._Loss
) -> torch.Tensor:
    """
    Computes the empirical NTK (Neural Tangent Kernel) matrix for a given batch of inputs.
    
    Args:
        model: The neural network model.
        inputs: A tensor of shape [batch_size, ...] containing the inputs.
        labels: A tensor of shape [batch_size, ...] containing the corresponding labels.
        criterion: The loss function (e.g., CrossEntropyLoss).
    
    Returns:
        ntk_matrix: A tensor of shape [batch_size, batch_size] representing the NTK.
    """
    model.train()
    batch_size = inputs.size(0)
    
    # Get all parameters that require gradients in a consistent order.
    parameters = [p for p in model.parameters() if p.requires_grad]
    n_params = sum(p.numel() for p in parameters)
    
    # Preallocate a Jacobian matrix of shape [batch_size, n_params]
    jacobian = torch.zeros(batch_size, n_params, device=inputs.device)
    
    for i in range(batch_size):
        model.zero_grad()
        x_i = inputs[i:i+1]  # Shape: [1, ...]
        y_i = labels[i:i+1]
        y_pred = model(x_i)
        loss = criterion(y_pred, y_i)
        loss.backward(retain_graph=True)
        
        idx = 0
        for param in parameters:
            if param.grad is None:
                raise RuntimeError(f"Gradient for parameter {param} is None.")
            grad_flat = param.grad.flatten()
            jacobian[i, idx:idx+grad_flat.numel()] = grad_flat
            idx += grad_flat.numel()
    
    ntk_matrix = jacobian @ jacobian.t()
    return ntk_matrix.detach()

# Example usage:
loss_fn = nn.CrossEntropyLoss()

# Compute NTK matrix using sample_input and sample_labels
ntk_matrix = compute_empirical_ntk(resnet18, sample_input, sample_labels, loss_fn)

print("NTK Matrix Shape:", ntk_matrix.shape)  # Should be (batch_size, batch_size)
display(ntk_matrix)

# SAE training

In [ ]:
#NOTE You might want to experiment with different choices for the nonlinearity or normalization to  
#see how they affect the downstream correlations.


from tqdm.auto import trange, tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# --- Sparse Autoencoder Module ---
class SparseAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, l1_lambda: float = 1e-6):
        """
        Args:
            input_dim: Dimension of the flattened input (e.g., 256*14*14).
            hidden_dim: Dimension of the latent space.
            l1_lambda: Coefficient for L1 sparsity penalty.
        """
        super(SparseAutoencoder, self).__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim)
        self.l1_lambda = l1_lambda
        
    def forward(self, x):
        # Use Leaky ReLU to avoid zeroing out negative activations entirely.
        latent = F.leaky_relu(self.encoder(x), negative_slope=0.01)
        reconstruction = self.decoder(latent)
        return latent, reconstruction
    
    def loss(self, x, reconstruction, latent):
        # Reconstruction loss (MSE)
        mse_loss = F.mse_loss(reconstruction, x)
        # L1 loss on the latent activations for sparsity
        l1_loss = torch.mean(torch.abs(latent))
        return mse_loss + self.l1_lambda * l1_loss

# --- Training Function ---
def train_sae(sae: SparseAutoencoder, activations: torch.Tensor, epochs: int = 10, batch_size: int = 4, lr: float = 1e-3):
    """
    Trains the sparse autoencoder on the given activations.
    
    Args:
        sae: The SparseAutoencoder model.
        activations: Tensor of shape [num_samples, channels, height, width].
        epochs: Number of training epochs.
        batch_size: Batch size for training.
        lr: Learning rate.
        
    Returns:
        The trained SAE model.
    """
    sae.train()
    optimizer = optim.Adam(sae.parameters(), lr=lr)
    
    num_samples = activations.size(0)
    # Flatten activations: shape [num_samples, channels * height * width]
    activations_flat = activations.view(num_samples, -1)
    
    dataset = torch.utils.data.TensorDataset(activations_flat)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    print_every = 50  # Set N: print every N epochs
    
    for epoch in trange(epochs, desc="Training Epochs"):
        epoch_loss = 0.0
        latent_means = []
        for batch in dataloader:
            x_batch = batch[0]
            optimizer.zero_grad()
            latent, reconstruction = sae(x_batch)
            loss = sae.loss(x_batch, reconstruction, latent)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * x_batch.size(0)
            latent_means.append(latent.mean().item())
        epoch_loss /= num_samples
        avg_latent = sum(latent_means) / len(latent_means)
        if (epoch + 1) % print_every == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.6f}, Avg Latent: {avg_latent:.6f}")
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.6f}, Avg Latent: {avg_latent:.6f}")

    
    return sae



# --- SAE Correlation Computation ---
def compute_sae_correlation(sae: SparseAutoencoder, activations: torch.Tensor) -> torch.Tensor:
    """
    Computes a correlation matrix from SAE latent representations.
    
    Args:
        sae: Trained SparseAutoencoder.
        activations: Input activations [num_samples, channels, height, width].
    
    Returns:
        correlation_matrix: Tensor of shape [num_samples, num_samples]
    """
    sae.eval()
    with torch.no_grad():
        num_samples = activations.size(0)
        activations_flat = activations.view(num_samples, -1)
        latent, _ = sae(activations_flat)
        correlation_matrix = torch.matmul(latent, latent.t())
    return correlation_matrix

# --- Main Experiment ---
# Assuming you already have layer4_activations from ResNet-18 with shape [batch_size, 256, 14, 14]
batch_size_val = layer4_activations.size(0)
input_dim = layer4_activations.size(1) * layer4_activations.size(2) * layer4_activations.size(3)
hidden_dim = 512  # Adjust if needed

# Instantiate and train the SAE using the best settings found (l1_lambda=1e-6, 10 epochs, batch_size=4)
sae_model = SparseAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim, l1_lambda=1e-6).to(device)
trained_sae = train_sae(sae_model, layer4_activations, epochs=100, batch_size=4, lr=1e-3)

# Compute the SAE correlation matrix (this variable is used in downstream plots)
sae_corr_matrix = compute_sae_correlation(trained_sae, layer4_activations)
print("SAE Correlation Matrix Shape:", sae_corr_matrix.shape)
display(sae_corr_matrix)

# visualize

In [ ]:

# # # import torch
# # # import matplotlib.pyplot as plt
# # # import seaborn as sns

# # # def normalize_matrix(M: torch.Tensor) -> torch.Tensor:
# # #     """
# # #     Normalizes a similarity matrix using diagonal normalization (cosine similarity).
    
# # #     Args:
# # #         M: Input matrix of shape [n, n].
        
# # #     Returns:
# # #         Normalized matrix of shape [n, n] with diagonal elements equal to 1.
# # #     """
# # #     diag = torch.sqrt(torch.diag(M))
# # #     norm_matrix = M / (diag.unsqueeze(1) * diag.unsqueeze(0) + 1e-8)
# # #     return norm_matrix

# # # def plot_heatmap(matrix: torch.Tensor, title: str, figsize: tuple=(6,5), cmap: str="viridis"):
# # #     plt.figure(figsize=figsize)
# # #     sns.heatmap(matrix.cpu().numpy(), annot=True, fmt=".2f", cmap=cmap)
# # #     plt.title(title)
# # #     plt.xlabel("Sample Index")
# # #     plt.ylabel("Sample Index")
# # #     plt.tight_layout()
# # #     plt.show()

# # # # Assume activation_similarity, ntk_matrix, and sae_corr_matrix have been computed as before.
# # # norm_activation_similarity = normalize_matrix(activation_similarity)
# # # norm_ntk_matrix = normalize_matrix(ntk_matrix)
# # # norm_sae_corr_matrix = normalize_matrix(sae_corr_matrix)

# # # print("Normalized Activation Similarity Matrix:")
# # # display(norm_activation_similarity)
# # # plot_heatmap(norm_activation_similarity, "Normalized Activation Similarity")

# # # print("Normalized NTK Matrix:")
# # # display(norm_ntk_matrix)
# # # plot_heatmap(norm_ntk_matrix, "Normalized NTK")

# # # print("Normalized SAE Correlation Matrix:")
# # # display(norm_sae_corr_matrix)
# # # plot_heatmap(norm_sae_corr_matrix, "Normalized SAE Correlation")




# # import torch
# # import matplotlib.pyplot as plt
# # import seaborn as sns

# # def normalize_matrix(M: torch.Tensor) -> torch.Tensor:
# #     """
# #     Normalizes a similarity matrix using diagonal normalization (cosine similarity).
    
# #     Args:
# #         M: Input matrix of shape [n, n].
        
# #     Returns:
# #         Normalized matrix of shape [n, n] with diagonal elements equal to 1.
# #     """
# #     diag = torch.sqrt(torch.diag(M))
# #     norm_matrix = M / (diag.unsqueeze(1) * diag.unsqueeze(0) + 1e-8)
# #     return norm_matrix

# # def plot_heatmap(matrix: torch.Tensor, title: str, figsize: tuple=(6,5), cmap: str="viridis"):
# #     plt.figure(figsize=figsize)
# #     # Force the colorbar limits to be from 0 to 1
# #     sns.heatmap(matrix.cpu().numpy(), annot=True, fmt=".2f", cmap=cmap, vmin=0, vmax=1)
# #     plt.title(title)
# #     plt.xlabel("Sample Index")
# #     plt.ylabel("Sample Index")
# #     plt.tight_layout()
# #     plt.show()

# # # Assume activation_similarity, ntk_matrix, and sae_corr_matrix have been computed as before.
# # norm_activation_similarity = normalize_matrix(activation_similarity)
# # norm_ntk_matrix = normalize_matrix(ntk_matrix)
# # norm_sae_corr_matrix = normalize_matrix(sae_corr_matrix)

# # print("Normalized Activation Similarity Matrix:")
# # display(norm_activation_similarity)
# # plot_heatmap(norm_activation_similarity, "Normalized Activation Similarity")

# # print("Normalized NTK Matrix:")
# # display(norm_ntk_matrix)
# # plot_heatmap(norm_ntk_matrix, "Normalized NTK")

# # print("Normalized SAE Correlation Matrix:")
# # display(norm_sae_corr_matrix)
# # plot_heatmap(norm_sae_corr_matrix, "Normalized SAE Correlation")



# import torch
# import matplotlib.pyplot as plt
# import seaborn as sns

# def normalize_matrix(M: torch.Tensor) -> torch.Tensor:
#     """
#     Normalizes a similarity matrix using diagonal normalization (cosine similarity).
    
#     Args:
#         M: Input matrix of shape [n, n].
        
#     Returns:
#         Normalized matrix of shape [n, n] with diagonal elements equal to 1.
#     """
#     diag = torch.sqrt(torch.diag(M))
#     norm_matrix = M / (diag.unsqueeze(1) * diag.unsqueeze(0) + 1e-8)
#     return norm_matrix

# def plot_heatmap(matrix: torch.Tensor, title: str, figsize: tuple = (6, 5), cmap: str = "viridis"):
#     plt.figure(figsize=figsize)
#     # Force the colorbar limits to be from 0 to 1
#     sns.heatmap(matrix.cpu().numpy(), annot=True, fmt=".2f", cmap=cmap)#, vmin=0, vmax=1)
#     plt.title(title)
#     plt.xlabel("Sample Index")
#     plt.ylabel("Sample Index")
#     plt.tight_layout()
#     plt.show()

# def run_similarity_analysis(activation_similarity: torch.Tensor, 
#                             ntk_matrix: torch.Tensor, 
#                             sae_corr_matrix: torch.Tensor,
#                             normalize: bool = True):
#     """
#     Wrapper function to process and display similarity matrices.
    
#     Args:
#         activation_similarity: Similarity matrix for activations.
#         ntk_matrix: Neural tangent kernel matrix.
#         sae_corr_matrix: SAE correlation matrix.
#         normalize: If True, apply normalization to matrices.
#     """
#     if normalize:
#         act_sim = normalize_matrix(activation_similarity)
#         ntk = normalize_matrix(ntk_matrix)
#         sae = normalize_matrix(sae_corr_matrix)
#     else:
#         act_sim = activation_similarity
#         ntk = ntk_matrix
#         sae = sae_corr_matrix

#     print("Activation Similarity Matrix:")
#     display(act_sim)
#     plot_heatmap(act_sim, "Activation Similarity")
    
#     print("NTK Matrix:")
#     display(ntk)
#     plot_heatmap(ntk, "NTK")
    
#     print("SAE Correlation Matrix:")
#     display(sae)
#     plot_heatmap(sae, "SAE Correlation")



# run_similarity_analysis(activation_similarity,
#                             ntk_matrix,
#                             sae_corr_matrix,
#                             normalize=True)



# run_similarity_analysis(activation_similarity,
#                             ntk_matrix,
#                             sae_corr_matrix,
#                             normalize=False)



import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def normalize_matrix(M: torch.Tensor) -> torch.Tensor:
    """
    Normalizes a similarity matrix using diagonal normalization (cosine similarity).
    
    Args:
        M: Input matrix of shape [n, n].
        
    Returns:
        Normalized matrix of shape [n, n] with diagonal elements equal to 1.
    """
    diag = torch.sqrt(torch.diag(M))
    norm_matrix = M / (diag.unsqueeze(1) * diag.unsqueeze(0) + 1e-8)
    return norm_matrix

def plot_heatmap(matrix: torch.Tensor, title: str, figsize: tuple = (6, 5), cmap: str = "viridis", custom_diag: bool = False):
    """
    Plots a heatmap of the given matrix.
    
    If custom_diag is True:
      - The color range is determined only by the off-diagonal values.
      - Diagonal cells are overlaid in white.
    
    Args:
        matrix: The input matrix (torch.Tensor).
        title: Title of the plot.
        figsize: Figure size.
        cmap: Colormap.
        custom_diag: Toggle to enable the special handling of the diagonal.
    """
    plt.figure(figsize=figsize)
    matrix_np = matrix.cpu().numpy()

    if custom_diag:
        # Get off-diagonal values for setting vmin and vmax.
        mask_off_diag = ~np.eye(matrix_np.shape[0], dtype=bool)
        off_diag_vals = matrix_np[mask_off_diag]
        vmin, vmax = off_diag_vals.min(), off_diag_vals.max()
    else:
        vmin, vmax = None, None

    ax = sns.heatmap(matrix_np, annot=True, fmt=".2f", cmap=cmap, vmin=vmin, vmax=vmax)
    plt.title(title)
    plt.xlabel("Sample Index")
    plt.ylabel("Sample Index")
    
    if custom_diag:
        # Overlay white boxes on the diagonal.
        for i in range(matrix_np.shape[0]):
            ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=True, color="white", lw=0))
    
    plt.tight_layout()
    plt.show()

def run_similarity_analysis(activation_similarity: torch.Tensor, 
                            ntk_matrix: torch.Tensor, 
                            sae_corr_matrix: torch.Tensor,
                            normalize: bool = True,
                            custom_diag: bool = False):
    """
    Processes and displays similarity matrices.
    
    Args:
        activation_similarity: Similarity matrix for activations.
        ntk_matrix: Neural tangent kernel matrix.
        sae_corr_matrix: SAE correlation matrix.
        normalize: If True, apply normalization to matrices.
        custom_diag: If True, limits the colorbar to off-diagonal correlations and overlays the diagonal in white.
    """
    if normalize:
        act_sim = normalize_matrix(activation_similarity)
        ntk = normalize_matrix(ntk_matrix)
        sae = normalize_matrix(sae_corr_matrix)
    else:
        act_sim = activation_similarity
        ntk = ntk_matrix
        sae = sae_corr_matrix

    print("Activation Similarity Matrix:")
    display(act_sim)
    plot_heatmap(act_sim, "Activation Similarity", custom_diag=custom_diag)
    
    print("NTK Matrix:")
    display(ntk)
    plot_heatmap(ntk, "NTK", custom_diag=custom_diag)
    
    print("SAE Correlation Matrix:")
    display(sae)
    plot_heatmap(sae, "SAE Correlation", custom_diag=custom_diag)


# Example usage:
run_similarity_analysis(activation_similarity,
                        ntk_matrix,
                        sae_corr_matrix,
                        normalize=True,
                        custom_diag=True)

run_similarity_analysis(activation_similarity,
                        ntk_matrix,
                        sae_corr_matrix,
                        normalize=False,
                        custom_diag=True)


